# Does refusal wobble predict model safety? — SPI Tier-0 demo

**Artifact:** *SPI — Safety Proximity Indicators*, a TIER-0 feasibility test of the
hypothesis that **safety = nearness to a tipping point**: if a safety-tuned model sits
close to a bifurcation between refusing and complying, then a *refusal observable*
`r_t` measured during ordinary sampled generation on **harmless prompts only** should
show classic early-warning signals — slow relaxation (small decay rate `lambda`),
inflated across-rollout variance `Var*`, high lag-1 autocorrelation `AC1`, and
flickering across the decision boundary.

The full run measured 4 models (Qwen3-0.6B base / instruct / abliterated + a SmolLM2-360M
anchor) x 20 harmless prompts x 20 paired rollouts x 192 generated steps on an RTX A4500
(94 min of GPU time). **That GPU arm cannot run in a 10-minute Colab notebook**, so this
demo does the two things that *can*:

1. **Re-runs the make-or-break arm live** — Stage H, the synthetic identifiability study.
   It asks the decisive question *before* any model result is believed: **is `lambda`
   recoverable at all** from a series of the achievable length and the *observed* noise
   level? This is the original `spi/validity.py` + `spi/indicators.py` code, fed the
   measured `noise_sd` and `amp` from the archived run, and it reproduces the archived
   grid cell for cell.
2. **Loads the archived measured results** and shows the two disconfirmations the
   artifact reports: `lambda` is **not identifiable** at any geometry reached, and a
   **random-direction control reproduces the panel ordering**.

**Headline: DISCONFIRMATION, twice over.** The pre-registered rule demands `T_fit >= 128`;
even there the requirement moves to `n_roll >= 40` against the achieved 20. And on the
only pair that isolates safety tuning (instruct vs abliterated) the *random* direction
separates (-0.493, CI excludes 0) while the *refusal* direction does not (-0.226, n.s.). Fluctuation indicators track **lineage,
not safety**.

## Setup

Install the dependencies. Everything this demo needs (`numpy`, `scipy`, `matplotlib`) is
pre-installed on Colab, so the installs are behind the `google.colab` guard and only fire
in a local/plain-Jupyter environment, pinned to Colab's exact versions.

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# numpy, scipy, matplotlib — pre-installed on Colab, install locally only (Colab's versions)
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'scipy==1.16.3', 'matplotlib==3.10.0')

### Imports

The original modules import `numpy`, `scipy.optimize` and `loguru`; `loguru` is only used
for progress logging, so the demo drops it and keeps everything else as in the artifact.
`matplotlib` is added for the final visualisation cell.

In [ ]:
from __future__ import annotations

import json
import os
import time
from typing import Any

import numpy as np
from scipy import optimize

import matplotlib
import matplotlib.pyplot as plt

### Data loading

`mini_demo_data.json` is the curated subset of the artifact's `method_out.json`: the
whole `synthetic_lambda_identifiability` dataset (96 archived grid cells), the observed
`noise_sd` / `amp` that drive the re-simulation, the archived panel-level indicators for
the 4 models, and the control verdicts.

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-c9546e-rating-model-safety-in-eighty-forward-pa/main/round-1/experiment-1/demo/mini_demo_data.json"
import json, os

REQUIRED = ("synthetic_meta", "model_level", "controls", "examples")

def _ok(d):  # guards against a stale/partial copy being served by the CDN
    return isinstance(d, dict) and all(k in d for k in REQUIRED)

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            d = json.loads(response.read().decode())
        if _ok(d): return d
    except Exception: pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f:
            d = json.load(f)
        if _ok(d): return d
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [ ]:
data = load_data()
print(data["method_name"])
print("verdict:", data["verdict"]["code"])
print("archived synthetic cells:", len(data["examples"]))
print("panel models:", [m["member"] + " (" + m["lineage"] + ")" for m in data["model_level"]])

## Config

All tunable parameters of the re-run in one place. The synthetic grid is the artifact's
own: 6 true decay rates x 4 fit lengths x 4 rollout counts = 96 cells, each Monte-Carlo'd
over `N_REPS` replicates.

`N_REPS` is the only knob that trades runtime for Monte-Carlo precision. The original run
used **500** replicates per cell; see the note in the cell below for what this notebook
actually uses and why.

In [ ]:
# ---- Monte-Carlo size -------------------------------------------------------
# The ORIGINAL full run used 500 — it fits the notebook budget (~4.5 min for the
# 96-cell grid), so the demo runs the study at full Monte-Carlo size. Drop to
# ~50 for a ~30 s pass if you just want to see it move.
N_REPS = 500          # replicates per grid cell

# ---- The pre-registered synthetic grid (unchanged from the artifact) --------
LAMBDAS = (0.02, 0.05, 0.1, 0.2, 0.5, 1.0)   # true decay rates to recover
T_FITS  = (16, 32, 64, 128)                  # fit-window lengths (steps)
N_ROLLS = (4, 12, 20, 40)                    # paired rollouts averaged
SEED    = 4242                               # artifact's seed for the study

# ---- Pre-registered acceptance rule for a (T_fit, n_roll) cell --------------
BIAS_TOL = 0.20   # |bias| < 0.20 * true_lambda
SD_TOL   = 0.50   # sd      < 0.50 * true_lambda

# ---- Noise level: MEASURED on the real models, read from the archive --------
SYN = data["synthetic_meta"]
NOISE_SD = SYN["noise_sd"]   # observed per-rollout residual sd of r_t
AMP      = SYN["amp"]        # observed perturbation amplitude |delta_0|

# ---- The geometry the real run actually achieved ----------------------------
ACHIEVED = data["controls"]["lambda_identifiable_at_achieved_geometry"]["achieved_geometry"]

print(f"grid = {len(LAMBDAS)}x{len(T_FITS)}x{len(N_ROLLS)} = "
      f"{len(LAMBDAS)*len(T_FITS)*len(N_ROLLS)} cells x {N_REPS} reps")
print(f"observed noise_sd={NOISE_SD:.5f}  amp={AMP:.5f}  (SNR={AMP/NOISE_SD:.2f})")
print("achieved real geometry:", ACHIEVED)
print("archived rule:", SYN["rule"]["note"])

## The `lambda` estimator (`spi/indicators.py`)

`fit_lambda_nls` is the PRIMARY estimator: a robust (soft-L1) non-linear least-squares fit
of `delta_t = A*exp(-lambda*t) + b`.

The `signed=True` path is **bug fix (c)** that the pre-flight gates caught. The
pre-registered statistic was `mean_j |delta_{t,j}|`, but `E|N(mu, sigma)| > |mu|`, so that
average is biased upward *and the bias does not vanish as rollouts are added* — it
converges to `E|X|`, not `|E X|`. Its tail flattens onto a `~0.8*sigma` floor whose
curvature the free offset `b` cannot absorb, which biases `lambda` upward by +38% to +68%
at every `n_roll`. The **signed** across-rollout mean is unbiased and its noise falls as
`sigma/sqrt(n_roll)`, so adding rollouts actually buys identifiability. Both are computed
here so the size of the correction is a reported number rather than an assertion.

`half_life_auc` is the pre-registered **substitute** statistic used when the rate fit is
not identifiable — assumption-free area under `|delta_t|` normalised by `|delta_1|`.

In [ ]:
def _exp_model(t: np.ndarray, A: float, lam: float, b: float) -> np.ndarray:
    return A * np.exp(-lam * t) + b


def fit_lambda_nls(d: np.ndarray, lam_bounds: tuple[float, float] = (1e-3, 2.0),
                   signed: bool = False) -> dict[str, Any]:
    """Estimator #1 (PRIMARY): robust NLS fit of delta_t = A*exp(-lam*t) + b.

    `signed=True` fits the SIGNED across-rollout mean deviation and lets A take
    either sign. That is the statistically correct target: mean_j |delta_{t,j}|
    is biased upward, because E|N(mu, sigma)| > |mu|, and — critically — the bias
    does NOT vanish as rollouts are added, since the average converges to E|X|
    rather than |E X|. Its tail therefore flattens onto a ~0.8*sigma floor whose
    curvature the free offset cannot absorb, which biases lambda upward. The
    signed mean is unbiased and its noise falls as sigma/sqrt(n_roll), so adding
    rollouts actually buys identifiability. `signed=False` reproduces the
    absolute-value statistic as the pre-registered secondary.
    """
    d = np.asarray(d, dtype=np.float64)
    t = np.arange(d.size, dtype=np.float64)
    ok = np.isfinite(d)
    if ok.sum() < 6:
        return {"lambda": None, "reason": "too_few_finite_points", "n": int(ok.sum())}
    t, d = t[ok], d[ok]
    tail = np.median(d[-max(3, d.size // 4):])
    b0 = float(tail)
    A0 = float(d[0] - tail)
    if signed:
        lo_A, hi_A = -np.inf, np.inf
        if abs(A0) < 1e-9:
            A0 = 1e-6
    else:
        lo_A, hi_A = 0.0, np.inf
        A0 = max(A0, 1e-6)
    try:
        popt, pcov = optimize.curve_fit(
            _exp_model, t, d,
            p0=[A0, 0.1, b0],
            bounds=([lo_A, lam_bounds[0], -np.inf], [hi_A, lam_bounds[1], np.inf]),
            loss="soft_l1", f_scale=max(float(np.std(d)), 1e-6), max_nfev=20000,
        )
    except Exception as exc:  # noqa: BLE001 - a failed fit must be null + reason
        return {"lambda": None, "reason": f"curve_fit_failed:{type(exc).__name__}"}
    A, lam, b = (float(v) for v in popt)
    pred = _exp_model(t, A, lam, b)
    ss_res = float(((d - pred) ** 2).sum())
    ss_tot = float(((d - d.mean()) ** 2).sum())
    r2 = 1.0 - ss_res / ss_tot if ss_tot > 0 else float("nan")
    se = float(np.sqrt(np.diag(pcov))[1]) if np.all(np.isfinite(pcov)) else float("nan")
    at_bound = lam <= lam_bounds[0] * 1.01 or lam >= lam_bounds[1] * 0.99
    return {
        "lambda": lam, "A": A, "b": b, "r2": r2, "se": se if np.isfinite(se) else None,
        "at_bound": bool(at_bound), "n": int(d.size), "reason": None,
    }


def half_life_auc(d: np.ndarray) -> dict[str, Any]:
    """PRE-REGISTERED SUBSTITUTE for lambda if the rate fit is not identifiable.

    Area under |delta_t| over the fit window, normalised by |delta_1|. This is a
    monotone proxy for 1/lambda and is far more robust than an exponential rate.
    Also reports the empirical half-life (first step where |delta| falls below
    half of |delta_1|).
    """
    d = np.asarray(d, dtype=np.float64)
    d = d[np.isfinite(d)]
    if d.size < 3 or not np.isfinite(d[0]) or abs(d[0]) < 1e-12:
        return {"auc_norm": None, "half_life": None, "reason": "degenerate_delta0"}
    auc = float(d.sum() / d[0])
    below = np.flatnonzero(d < 0.5 * d[0])
    hl = float(below[0]) if below.size else float(d.size)
    return {"auc_norm": auc, "half_life": hl, "delta_0": float(d[0]), "reason": None}

### Estimator correctness gate (T5)

Run **before** the study, exactly as in the artifact: (a) noiseless exponentials must be
recovered within 2%; (b) pure noise must **not** yield a confident number. A study whose
estimator fails this gate would produce confident nonsense.

In [ ]:
def estimator_unit_tests() -> dict[str, Any]:
    """T5 correctness gate — run BEFORE the study, on placeholder inputs.

    (a) noiseless exponentials must be recovered within 2%;
    (b) pure noise must NOT yield a confident number.
    """
    out: dict[str, Any] = {"noiseless": [], "pure_noise": []}
    for lam in (0.05, 0.1, 0.3, 0.8):
        t = np.arange(64, dtype=np.float64)
        d = 1.0 * np.exp(-lam * t) + 0.0
        fit = fit_lambda_nls(d)
        est = fit.get("lambda")
        rel = abs(est - lam) / lam if est is not None else None
        out["noiseless"].append({
            "true": lam, "est": est, "rel_err": rel,
            "within_2pct": bool(rel is not None and rel < 0.02),
        })
    rng = np.random.default_rng(0)
    for i in range(20):
        d = np.abs(rng.normal(0.0, 1.0, size=64))
        fit = fit_lambda_nls(d)
        out["pure_noise"].append({
            "lambda": fit.get("lambda"), "r2": fit.get("r2"),
            "at_bound": fit.get("at_bound"), "reason": fit.get("reason"),
        })
    out["noiseless_all_pass"] = all(x["within_2pct"] for x in out["noiseless"])
    r2s = [x["r2"] for x in out["pure_noise"] if x["r2"] is not None]
    out["pure_noise_median_r2"] = float(np.median(r2s)) if r2s else None
    out["pure_noise_flagged_rate"] = float(
        np.mean([bool(x["at_bound"]) or x["lambda"] is None or (x["r2"] or 0) < 0.2
                 for x in out["pure_noise"]])
    )
    return out


gate = estimator_unit_tests()
for x in gate["noiseless"]:
    print(f"  noiseless lambda={x['true']:<5} est={x['est']:.6f} "
          f"rel_err={x['rel_err']:.2e} within_2pct={x['within_2pct']}")
print("noiseless_all_pass   :", gate["noiseless_all_pass"])
print("pure_noise median r2 :", round(gate["pure_noise_median_r2"], 4))
print("pure_noise flagged   :", gate["pure_noise_flagged_rate"])
assert gate["noiseless_all_pass"], "T5 gate FAILED — estimator is not trustworthy"

## Stage H — the synthetic identifiability study (`spi/validity.py`)

One replicate simulates exactly what the real estimator consumes: per rollout, a decaying
signal `amp * exp(-lambda*t)` plus independent Gaussian noise at the **observed** sd. The
signed across-rollout mean is fitted (primary), and the absolute-value mean is simulated
alongside so its bias is measured rather than assumed.

A cell **passes** only if `|bias| < 0.20*lambda` **and** `sd < 0.50*lambda` — pre-registered.

In [ ]:
def simulate_delta_curve(true_lambda: float, amp: float, noise_sd: float,
                         T_fit: int, n_roll: int, rng: np.random.Generator
                         ) -> tuple[np.ndarray, np.ndarray]:
    """One replicate of the curves the real estimator consumes: (signed, abs).

    Per rollout the deviation is a decaying signal plus independent noise. The
    signed across-rollout mean is what the primary estimator fits; the
    absolute-value mean is simulated alongside so the study measures the bias
    that statistic carries rather than assuming it.
    """
    t = np.arange(T_fit, dtype=np.float64)
    signal = amp * np.exp(-true_lambda * t)
    noise = rng.normal(0.0, noise_sd, size=(T_fit, n_roll))
    per_rollout = signal[:, None] + noise
    return per_rollout.mean(axis=1), np.abs(per_rollout).mean(axis=1)


def _cell_worker(args: tuple) -> dict[str, Any]:
    true_lambda, amp, noise_sd, T_fit, n_roll, n_reps, seed = args
    rng = np.random.default_rng(seed)
    lams: list[float] = []
    lams_abs: list[float] = []
    aucs: list[float] = []
    n_fail = 0
    n_bound = 0
    for _ in range(n_reps):
        ds, da = simulate_delta_curve(true_lambda, amp, noise_sd, T_fit, n_roll, rng)
        fit = fit_lambda_nls(ds, signed=True)
        fit_a = fit_lambda_nls(da, signed=False)
        if fit_a.get("lambda") is not None:
            lams_abs.append(float(fit_a["lambda"]))
        if fit.get("lambda") is None:
            n_fail += 1
            continue
        if fit.get("at_bound"):
            n_bound += 1
        lams.append(float(fit["lambda"]))
        a = half_life_auc(np.abs(ds))
        if a.get("auc_norm") is not None:
            aucs.append(float(a["auc_norm"]))
    arr = np.asarray(lams, dtype=np.float64)
    arr_abs = np.asarray(lams_abs, dtype=np.float64)
    if arr.size < 10:
        return {
            "true_lambda": true_lambda, "T_fit": T_fit, "n_roll": n_roll,
            "n_ok": int(arr.size), "n_fail": n_fail, "bias": None, "sd": None,
            "passes": False, "reason": "insufficient_successful_fits",
        }
    bias = float(arr.mean() - true_lambda)
    sd = float(arr.std(ddof=1))
    # Bootstrap-percentile coverage of the true value across replicates.
    lo, hi = np.percentile(arr, [2.5, 97.5])
    passes = abs(bias) < BIAS_TOL * true_lambda and sd < SD_TOL * true_lambda
    return {
        "true_lambda": float(true_lambda), "T_fit": int(T_fit), "n_roll": int(n_roll),
        "amp": float(amp), "noise_sd": float(noise_sd),
        "n_ok": int(arr.size), "n_fail": int(n_fail), "n_at_bound": int(n_bound),
        "mean_est": float(arr.mean()), "median_est": float(np.median(arr)),
        "bias": bias, "rel_bias": float(bias / true_lambda), "sd": sd,
        "rel_sd": float(sd / true_lambda),
        "pct_2_5": float(lo), "pct_97_5": float(hi),
        "covers_truth": bool(lo <= true_lambda <= hi),
        "auc_mean": float(np.mean(aucs)) if aucs else None,
        "auc_sd": float(np.std(aucs, ddof=1)) if len(aucs) > 1 else None,
        # The pre-registered mean-|delta| statistic, measured side by side so the
        # size of its upward bias is a reported number, not an assertion.
        "abs_statistic_rel_bias": (
            float((arr_abs.mean() - true_lambda) / true_lambda) if arr_abs.size >= 10 else None),
        "abs_statistic_rel_sd": (
            float(arr_abs.std(ddof=1) / true_lambda) if arr_abs.size >= 10 else None),
        "passes": bool(passes), "reason": None,
    }

### Run the grid

The artifact fans the 96 cells over a 16-worker process pool; in the notebook they run
sequentially with a progress line, which keeps the code readable and stays well inside the
runtime budget. Seeds are assigned in the same order as the original
(`seed, seed+1, ...` over `lambda x T_fit x n_roll`), so the cells are directly comparable
to the archived ones.

In [ ]:
def synthetic_ar1_study(noise_sd: float, amp: float, *,
                        lambdas: tuple[float, ...] = LAMBDAS,
                        T_fits: tuple[int, ...] = T_FITS,
                        n_rolls: tuple[int, ...] = N_ROLLS,
                        n_reps: int = N_REPS, seed: int = SEED) -> dict[str, Any]:
    """Full grid. Returns the table plus the derived minimum-geometry rule."""
    jobs = []
    s = seed
    for lam in lambdas:
        for T_fit in T_fits:
            for n_roll in n_rolls:
                jobs.append((lam, amp, noise_sd, T_fit, n_roll, n_reps, s))
                s += 1
    print(f"Synthetic AR(1) study: {len(jobs)} cells x {n_reps} reps "
          f"(noise_sd={noise_sd:.4f}, amp={amp:.4f})")
    rows = []
    t0 = time.time()
    for i, job in enumerate(jobs, 1):
        rows.append(_cell_worker(job))
        if i % 12 == 0 or i == len(jobs):
            print(f"  {i:3d}/{len(jobs)} cells  ({time.time()-t0:5.1f}s)")
    rule = derive_min_geometry(rows, lambdas)
    print("Minimum-geometry rule:", rule["note"])
    return {"table": rows, "rule": rule, "n_reps": n_reps,
            "noise_sd": float(noise_sd), "amp": float(amp),
            "bias_tol": BIAS_TOL, "sd_tol": SD_TOL}


def derive_min_geometry(rows: list[dict[str, Any]],
                        lambdas: tuple[float, ...]) -> dict[str, Any]:
    """Smallest (T_fit, n_roll) cell passing the rule across the WHOLE lambda range.

    If no cell passes, that is the artifact's headline finding and is reported
    as such — never dressed up.
    """
    by_geom: dict[tuple[int, int], list[dict[str, Any]]] = {}
    for r in rows:
        by_geom.setdefault((r["T_fit"], r["n_roll"]), []).append(r)
    passing = []
    for (T_fit, n_roll), cells in by_geom.items():
        if len(cells) < len(lambdas):
            continue
        if all(c["passes"] for c in cells):
            passing.append((T_fit, n_roll))
    # Per-lambda relaxation: which lambdas are recoverable at the largest geometry.
    largest = max(by_geom, key=lambda k: (k[0], k[1]))
    per_lambda = {
        str(c["true_lambda"]): bool(c["passes"]) for c in by_geom[largest]
    }
    if not passing:
        return {
            "any_cell_passes": False,
            "min_T_fit": None, "min_n_roll": None,
            "per_lambda_at_largest_geometry": per_lambda,
            "largest_geometry": {"T_fit": largest[0], "n_roll": largest[1]},
            "note": (
                "NO (T_fit, n_roll) cell meets |bias| < 0.2*lambda AND sd < 0.5*lambda "
                "across the full lambda range. Under the pre-registered rule, lambda is "
                "reported with identifiable=false and the AUC/half-life substitute "
                "becomes the headline recovery statistic."
            ),
        }
    passing.sort(key=lambda k: (k[0], k[1]))
    T_min, n_min = passing[0]
    return {
        "any_cell_passes": True, "min_T_fit": int(T_min), "min_n_roll": int(n_min),
        "n_passing_cells": len(passing),
        "per_lambda_at_largest_geometry": per_lambda,
        "largest_geometry": {"T_fit": largest[0], "n_roll": largest[1]},
        "note": (
            f"lambda is reported as identifiable only at T_fit >= {T_min} and "
            f"n_roll >= {n_min} (pre-registered rule)."
        ),
    }


def is_identifiable(rule: dict[str, Any], T_fit: int, n_roll: int) -> bool:
    """Apply the pre-registered rule to a real measurement's geometry."""
    if not rule.get("any_cell_passes"):
        return False
    return T_fit >= int(rule["min_T_fit"]) and n_roll >= int(rule["min_n_roll"])


t0 = time.time()
syn = synthetic_ar1_study(NOISE_SD, AMP)
print(f"study wall-clock: {time.time()-t0:.1f}s")

### Apply the pre-registered rule to the geometry the real run achieved

This is the primary verdict of the artifact. The rule is derived from the simulation
above — it is not chosen after seeing the model results.

In [ ]:
rule = syn["rule"]
ident = is_identifiable(rule, ACHIEVED["T_fit"], ACHIEVED["n_roll"])
print("derived rule      :", {k: rule[k] for k in ("any_cell_passes", "min_T_fit", "min_n_roll")})
print("achieved geometry :", ACHIEVED)
print("lambda identifiable at the achieved geometry?", ident)
print()
print("archived rule     :", {k: SYN["rule"][k] for k in ("any_cell_passes", "min_T_fit", "min_n_roll")})
print("archived verdict  :", data["verdict"]["code"])
print()
print("per-lambda recovery at the largest geometry",
      rule["largest_geometry"], ":")
for lam, ok in rule["per_lambda_at_largest_geometry"].items():
    print(f"   lambda={lam:<5} passes={ok}")

### Does the live re-run agree with the archive?

Each re-simulated cell is matched to its archived counterpart on `(lambda, T_fit, n_roll)`
and the pass/fail decision compared. With `N_REPS` equal to the original 500 and the same
seed schedule the two should agree on essentially every cell; any residual difference is
Monte-Carlo noise near the tolerance boundary, which the printout makes explicit.

In [ ]:
def parse_archived(ex: dict) -> dict:
    """Archived cells store their numbers in formatted strings; pull them back out."""
    head = dict(kv.split("=") for kv in
                [p.strip() for p in ex["input"].split("|")][:3])
    sgn = dict(kv.split("=") for kv in
               [p.strip() for p in ex["predict_our_method_signed_estimator"].split(";")])
    ab = dict(kv.split("=") for kv in
              [p.strip() for p in ex["predict_baseline_abs_estimator"].split(";")])
    return {
        "true_lambda": float(head["true_lambda"]), "T_fit": int(head["T_fit"]),
        "n_roll": int(head["n_roll"]),
        "rel_bias": float(sgn["rel_bias"]), "rel_sd": float(sgn["rel_sd"]),
        "passes": sgn["passes"] == "True",
        "abs_rel_bias": float(ab["rel_bias"]), "abs_rel_sd": float(ab["rel_sd"]),
    }


arch = {(a["true_lambda"], a["T_fit"], a["n_roll"]): a
        for a in (parse_archived(e) for e in data["examples"])}

agree = disagree = 0
rows_cmp = []
for r in syn["table"]:
    key = (r["true_lambda"], r["T_fit"], r["n_roll"])
    a = arch.get(key)
    if a is None:
        continue
    same = (r["passes"] == a["passes"])
    agree += same
    disagree += (not same)
    rows_cmp.append((key, r, a, same))

print(f"cells compared: {agree + disagree}   pass/fail AGREE: {agree}   DISAGREE: {disagree}")
if disagree:
    print("\ndisagreeing cells (Monte-Carlo noise near the tolerance boundary):")
    for key, r, a, same in rows_cmp:
        if not same:
            print(f"  lambda={key[0]:<5} T_fit={key[1]:<4} n_roll={key[2]:<3} "
                  f"live passes={r['passes']} (rel_bias={r['rel_bias']:+.3f}, rel_sd={r['rel_sd']:.3f}) | "
                  f"archived passes={a['passes']} (rel_bias={a['rel_bias']:+.3f}, rel_sd={a['rel_sd']:.3f})")

### Bug fix (c): how big is the mean-|delta| bias, really?

The pre-registered statistic was `mean_j |delta_{t,j}|`. Every cell measured both, so the
upward bias it carries is a number, not an argument. The comparison below is restricted to
cells where the signed estimator is well-behaved, so it isolates the statistic rather than
the regime.

In [ ]:
good = [r for r in syn["table"] if r["passes"]]
if good:
    sg = np.array([r["rel_bias"] for r in good])
    ab = np.array([r["abs_statistic_rel_bias"] for r in good])
    print(f"cells where the SIGNED estimator passes: {len(good)}")
    print(f"  signed   relative bias: median {np.median(sg):+.4f}   "
          f"range [{sg.min():+.4f}, {sg.max():+.4f}]")
    print(f"  abs      relative bias: median {np.median(ab):+.4f}   "
          f"range [{ab.min():+.4f}, {ab.max():+.4f}]")
else:
    print("no passing cells at this N_REPS")

## Archived measured results — the two disconfirmations

Nothing below is re-simulated; these are the numbers from the 94-minute GPU run.

**(1) Panel validity passes** — the panel really does span the safety axis
(instruct refuses 22.5% of plain-harmful prompts, abliterated 0.0%), so a null result is
informative rather than an artifact of a degenerate panel.

**(2) Indicators track lineage, not safety** — the three Qwen3-0.6B members overlap on
`Var*` and `AC1` while the SmolLM2 anchor separates, and the pre-registered ordering
partly *reverses* (instruct has the LOWEST `Var*` of the triad).

**(3) The random-direction control** — a random unit vector at the same layer and magnitude
separates the panel as well as the refusal direction does. The artifact's supplementary
verdict `CONTROL_REPRODUCES_ORDERING_GENERIC_MIXING` rests on the significance-count
comparison (2/3 vs 2/3 significant, and on the only pair isolating safety tuning the
control separates while the treatment does not). The pre-registered boolean printed below
is a *stricter* flag — exact ordering reproduction at the layer-L readout — and it is
`False`; both are reported here rather than conflated. The per-model `lam_random` column
in the table above is the raw control measurement.

In [ ]:
pv = data["panel_validity"]
print("PANEL VALIDITY")
print(f"  instruct harmful-refusal    : {pv['instruct_harmful_refusal']:.3f}")
print(f"  abliterated harmful-refusal : {pv['abliterated_harmful_refusal']:.3f}")
print(f"  base harmful-refusal        : {pv['base_harmful_refusal']:.3f}")
print(f"  criterion: {pv['criterion']}  ->  panel_valid={pv['panel_valid']}")
print()

print("PER-MODEL INDICATORS (archived, label-free, 0 harmful prompts)")
hdr = f"{'model':<28}{'Var*':>8}{'AC1':>8}{'lam_refuse':>12}{'lam_random':>12}"
print(hdr); print("-" * len(hdr))
for m in data["model_level"]:
    i = m["indicators"]
    print(f"{m['lineage'] + '/' + m['member']:<28}"
          f"{i['var_star']:>8.3f}{i['ac1']:>8.3f}"
          f"{i['lambda_toward_refuse']:>12.3f}{i['lambda_random_direction']:>12.3f}")
print()

print("SPI vs the SUPERVISED baselines (which are handed the 32 harmful prompts SPI is denied)")
for m in data["model_level"]:
    print(f"  {m['lineage'] + '/' + m['member']:<28} {m['spi']}")
    print(f"  {'':<28} {m['baseline_diffmeans_auroc']}")
    print(f"  {'':<28} ground truth: {m['output']}")
print()

for name, c in data["controls"].items():
    print(f"CONTROL {name}: value={c['value']}")

## Results

Four panels:

1. **Identifiability heat map** — `rel_sd` of the signed `lambda` estimate across the
   `(T_fit, n_roll)` grid, pooled over the 6 true rates. Cells that pass the pre-registered
   rule for *every* rate are outlined; the geometry the real run achieved is marked. The
   whole disconfirmation is visible here: the achieved cell is not in the passing set.
2. **Bias vs `n_roll`** — signed vs absolute statistic, showing that the absolute-value
   bias does *not* shrink as rollouts are added while the signed one does.
3. **Live vs archived** — `rel_sd` of the re-run against the archive, cell by cell.
4. **Panel indicators** — `Var*` per model, coloured by lineage, with the archived
   harmful-refusal rate annotated: the Qwen triad clusters together regardless of safety
   tuning.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13.5, 10))

# --- (1) identifiability heat map ------------------------------------------
ax = axes[0, 0]
grid = np.full((len(T_FITS), len(N_ROLLS)), np.nan)
allpass = np.zeros_like(grid, dtype=bool)
for i, T in enumerate(T_FITS):
    for j, n in enumerate(N_ROLLS):
        cells = [r for r in syn["table"] if r["T_fit"] == T and r["n_roll"] == n]
        vals = [r["rel_sd"] for r in cells if r.get("rel_sd") is not None]
        if vals:
            grid[i, j] = float(np.median(vals))
        allpass[i, j] = bool(cells) and all(c["passes"] for c in cells)
im = ax.imshow(np.log10(grid), cmap="viridis_r", aspect="auto")
ax.set_xticks(range(len(N_ROLLS)), [str(n) for n in N_ROLLS])
ax.set_yticks(range(len(T_FITS)), [str(t) for t in T_FITS])
ax.set_xlabel("n_roll (paired rollouts)"); ax.set_ylabel("T_fit (steps)")
ax.set_title("median rel_sd of signed lambda (log10)\noutline = passes rule for ALL lambdas")
for i in range(len(T_FITS)):
    for j in range(len(N_ROLLS)):
        if allpass[i, j]:
            ax.add_patch(plt.Rectangle((j - .5, i - .5), 1, 1, fill=False,
                                       edgecolor="crimson", lw=3))
        ax.text(j, i, f"{grid[i, j]:.2f}", ha="center", va="center",
                color="w", fontsize=9)
ai = T_FITS.index(ACHIEVED["T_fit"]); aj = N_ROLLS.index(ACHIEVED["n_roll"])
ax.plot(aj, ai, marker="X", ms=16, color="red", mec="k", mew=1.2,
        label="achieved by the real run")
ax.legend(loc="lower left", fontsize=8)
fig.colorbar(im, ax=ax, shrink=.8)

# --- (2) signed vs absolute statistic bias ---------------------------------
ax = axes[0, 1]
for stat, key, style in (("signed (fixed)", "rel_bias", "-o"),
                         ("mean-|delta| (pre-registered)", "abs_statistic_rel_bias", "--s")):
    ys = []
    for n in N_ROLLS:
        vals = [r[key] for r in syn["table"]
                if r["n_roll"] == n and r["T_fit"] == max(T_FITS) and r.get(key) is not None]
        ys.append(float(np.median(vals)) if vals else np.nan)
    ax.plot(N_ROLLS, ys, style, label=stat)
ax.axhline(0, color="k", lw=.8)
ax.axhspan(-BIAS_TOL, BIAS_TOL, color="green", alpha=.12, label=f"|bias| tolerance {BIAS_TOL}")
ax.set_xscale("log"); ax.set_xticks(N_ROLLS, [str(n) for n in N_ROLLS])
ax.set_yscale("symlog", linthresh=1.0)
ax.set_xlabel("n_roll"); ax.set_ylabel("median relative bias")
ax.set_title(f"bug fix (c): absolute-value bias does not\nvanish with rollouts (T_fit={max(T_FITS)})")
ax.legend(fontsize=8)

# --- (3) live re-run vs archive --------------------------------------------
ax = axes[1, 0]
lx = [a["rel_sd"] for _, r, a, _ in rows_cmp]
ly = [r["rel_sd"] for _, r, a, _ in rows_cmp]
cols = ["tab:green" if s else "tab:red" for _, _, _, s in rows_cmp]
ax.scatter(lx, ly, c=cols, s=26, alpha=.85, edgecolor="k", linewidth=.3)
lim = [min(lx + ly) * .8 + 1e-6, max(lx + ly) * 1.2]
ax.plot(lim, lim, "k--", lw=.8)
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("archived rel_sd"); ax.set_ylabel("live re-run rel_sd")
ax.set_title(f"reproduction: {agree}/{agree + disagree} cells agree on pass/fail\n"
             f"(green = same verdict, N_REPS={N_REPS})")

# --- (4) archived panel indicators -----------------------------------------
ax = axes[1, 1]
names = [m["lineage"] + "/" + m["member"] for m in data["model_level"]]
vals = [m["indicators"]["var_star"] for m in data["model_level"]]
lin = [m["lineage"] for m in data["model_level"]]
palette = {l: c for l, c in zip(sorted(set(lin)), ["tab:blue", "tab:orange"])}
bars = ax.bar(range(len(names)), vals, color=[palette[l] for l in lin])
for k, m in enumerate(data["model_level"]):
    rate = m["output"].split(";")[0].split("=")[1].split(" ")[0]
    ax.text(k, vals[k] + .05, f"refusal\n{rate}", ha="center", fontsize=8)
ax.set_xticks(range(len(names)), names, rotation=20, ha="right", fontsize=8)
ax.set_ylabel("Var* (across-rollout variance)")
ax.set_title("archived: Var* tracks LINEAGE, not safety\n(the Qwen triad overlaps; SmolLM2 separates)")
ax.set_ylim(0, max(vals) * 1.25)

fig.suptitle("SPI Tier-0 — lambda is not identifiable at any geometry reached", fontsize=13)
fig.tight_layout()
plt.show()

### Summary

In [ ]:
print("=" * 74)
print("SPI TIER-0 — DEMO SUMMARY")
print("=" * 74)
print(f"live re-run    : {len(syn['table'])} grid cells x {N_REPS} reps at the OBSERVED noise")
print(f"derived rule   : {rule['note']}")
print(f"achieved       : T_fit={ACHIEVED['T_fit']}, n_roll={ACHIEVED['n_roll']}"
      f"  ->  identifiable = {ident}")
print(f"reproduction   : {agree}/{agree + disagree} cells match the archived pass/fail verdict")
print()
print(f"archived verdict (pre-registered)  : {data['verdict']['code']}")
print(f"random-direction control reproduces ordering : "
      f"{data['controls']['random_direction_reproduces_ordering']['value']}")
print(f"panel valid                        : {data['panel_validity']['panel_valid']}")
print()
print("Conclusion: the fluctuation indicators are real and measurable, but at this model")
print("scale and series length lambda cannot be pinned down, and what separation exists is")
print("generic mixing that a random direction reproduces — not a safety signal.")